<a href="https://colab.research.google.com/github/Ay0-Belajar/Python-Data-Science/blob/Plotly/plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# model ML dengan Supervised Learning
kasus : prediksi diagnosa penyakit jantung

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import plotly.express as px
import plotly.graph_objects as go

# 1. Membuat dataset sintetis untuk prediksi penyakit jantung
np.random.seed(42)
num_samples = 1000

data_ml = {
    'age': np.random.randint(30, 75, num_samples),
    'sex': np.random.choice([0, 1], num_samples, p=[0.4, 0.6]), # 0=Female, 1=Male
    'cholesterol': np.random.randint(150, 300, num_samples),
    'blood_pressure': np.random.randint(100, 180, num_samples),
    'max_hr': np.random.randint(120, 200, num_samples),
    'exercise_angina': np.random.choice([0, 1], num_samples, p=[0.7, 0.3]), # 0=No, 1=Yes
}

df_ml = pd.DataFrame(data_ml)

# Membuat variabel target 'heart_disease' berdasarkan aturan yang disimulasikan
# Misal: usia lebih tinggi, kolesterol, tekanan darah, dan angina saat berolahraga meningkatkan risiko
heart_disease_risk = (
    0.3 * (df_ml['age'] / df_ml['age'].max()) +
    0.2 * df_ml['sex'] +
    0.25 * (df_ml['cholesterol'] / df_ml['cholesterol'].max()) +
    0.2 * (df_ml['blood_pressure'] / df_ml['blood_pressure'].max()) +
    0.1 * df_ml['exercise_angina'] -
    0.15 * (df_ml['max_hr'] / df_ml['max_hr'].max()) # Denyut jantung maksimal yang lebih tinggi mungkin menurunkan risiko
)
df_ml['heart_disease'] = (heart_disease_risk > np.percentile(heart_disease_risk, 60)).astype(int) # 40% teratas memiliki penyakit jantung

# Mengganti nama kolom ke Bahasa Indonesia
df_ml = df_ml.rename(columns={
    'age': 'usia',
    'sex': 'jenis_kelamin',
    'cholesterol': 'kolesterol',
    'blood_pressure': 'tekanan_darah',
    'max_hr': 'detak_jantung_maks',
    'exercise_angina': 'angina_saat_olahraga',
    'heart_disease': 'penyakit_jantung'
})

print("Dataset Sintetis Penyakit Jantung :")
display(df_ml.head())

# Klasifikasi (Classification) Digunakan ketika output yang ingin diprediksi berupa kategori atau label diskrit.
# 2. Membagi data menjadi training dan testing sets
X = df_ml.drop('penyakit_jantung', axis=1)
y = df_ml['penyakit_jantung']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Melatih model Machine Learning sederhana (Regresi Logistik)
model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
model.fit(X_train, y_train)

# 4. Melakukan prediksi pada test set
y_pred = model.predict(X_test)

# 5. Mengevaluasi model - Confusion Matrix dan Akurasi
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAkurasi Model: {accuracy:.2f}")
print("Confusion Matrix:")
print(cm)

# 6. Memvisualisasikan Confusion Matrix menggunakan Plotly
cm_labels = ['Tidak Sakit Jantung', 'Sakit Jantung']

fig_ml_cm = px.imshow(
    cm,
    x=cm_labels,
    y=cm_labels,
    color_continuous_scale='Blues',
    text_auto=True,
    labels=dict(x="Prediksi Model", y="Kondisi Aktual Pasien"),
    title="<b>Confusion Matrix Prediksi Penyakit Jantung</b>"
)

fig_ml_cm.update_layout(
    xaxis_title="Hasil Prediksi Model",
    yaxis_title="Kondisi Nyata Pasien"
)

fig_ml_cm.show()

# Memvisualisasikan Koefisien Fitur (Feature Importance) untuk Regresi Logistik
feature_importance = pd.DataFrame({
    'Fitur': X.columns,
    'Koefisien': model.coef_[0]
}).sort_values(by='Koefisien', ascending=False)

fig_feature_imp = px.bar(
    feature_importance,
    x='Fitur',
    y='Koefisien',
    color='Koefisien',
    color_continuous_scale=px.colors.sequential.RdBu,
    title='<b>Koefisien Fitur Model Regresi Logistik (Pengaruh pada Prediksi Sakit Jantung)</b>',
    labels={'Koefisien': 'Pengaruh pada Prediksi Sakit Jantung'}
)
fig_feature_imp.update_layout(showlegend=False)
fig_feature_imp.show()

Dataset Sintetis Penyakit Jantung :


,usia,jenis_kelamin,kolesterol,tekanan_darah,detak_jantung_maks,angina_saat_olahraga,penyakit_jantung
0,68,1,273,151,167,1,1
1,58,1,212,119,125,0,1
2,44,0,265,131,175,1,0
3,72,1,211,113,178,0,1
4,37,0,156,149,148,1,0



Akurasi Model: 0.90
Confusion Matrix:
[[157  23]
 [  8 112]]


## Prediksi Diagnosis Penyakit Jantung untuk Pasien Baru

Sekarang, setelah model dilatih, kita bisa menggunakan model tersebut untuk memprediksi diagnosis penyakit jantung untuk data pasien baru. Anda bisa mengubah nilai-nilai input di bawah untuk melihat bagaimana model membuat prediksi.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

def predict_new_patient_heart_disease(model, patient_data: dict, feature_columns: list) -> str:
    # Pastikan data pasien baru memiliki kolom yang sama dengan data pelatihan
    new_patient_df = pd.DataFrame([patient_data], columns=feature_columns)

    # Lakukan prediksi
    prediction = model.predict(new_patient_df)
    prediction_proba = model.predict_proba(new_patient_df)

    result = ""
    if prediction[0] == 1:
        result = f"Pasien diprediksi memiliki **Penyakit Jantung** (Probabilitas: {prediction_proba[0][1]*100:.2f}%)"
    else:
        result = f"Pasien diprediksi **Tidak Memiliki Penyakit Jantung** (Probabilitas: {prediction_proba[0][0]*100:.2f}%)"

    return result

# --- Re-defining dependencies from previous cell for independent execution ---
# 1. Membuat dataset sintetis untuk prediksi penyakit jantung
np.random.seed(42)
num_samples = 1000

data_ml = {
    'age': np.random.randint(30, 75, num_samples),
    'sex': np.random.choice([0, 1], num_samples, p=[0.4, 0.6]), # 0=Female, 1=Male
    'cholesterol': np.random.randint(150, 300, num_samples),
    'blood_pressure': np.random.randint(100, 180, num_samples),
    'max_hr': np.random.randint(120, 200, num_samples),
    'exercise_angina': np.random.choice([0, 1], num_samples, p=[0.7, 0.3]), # 0=No, 1=Yes
}

df_ml = pd.DataFrame(data_ml)

# Membuat variabel target 'heart_disease' berdasarkan aturan yang disimulasikan
heart_disease_risk = (
    0.3 * (df_ml['age'] / df_ml['age'].max()) +
    0.2 * df_ml['sex'] +
    0.25 * (df_ml['cholesterol'] / df_ml['cholesterol'].max()) +
    0.2 * (df_ml['blood_pressure'] / df_ml['blood_pressure'].max()) +
    0.1 * df_ml['exercise_angina'] -
    0.15 * (df_ml['max_hr'] / df_ml['max_hr'].max())
)
df_ml['heart_disease'] = (heart_disease_risk > np.percentile(heart_disease_risk, 60)).astype(int)

# Mengganti nama kolom ke Bahasa Indonesia
df_ml = df_ml.rename(columns={
    'age': 'usia',
    'sex': 'jenis_kelamin',
    'cholesterol': 'kolesterol',
    'blood_pressure': 'tekanan_darah',
    'max_hr': 'detak_jantung_maks',
    'exercise_angina': 'angina_saat_olahraga',
    'heart_disease': 'penyakit_jantung'
})

# 2. Membagi data menjadi training dan testing sets
X = df_ml.drop('penyakit_jantung', axis=1)
y = df_ml['penyakit_jantung']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Melatih model Machine Learning sederhana (Regresi Logistik)
model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
model.fit(X_train, y_train)
# --- End of dependencies re-definition ---

# Contoh data pasien baru
# Mengganti nama fitur ke Bahasa Indonesia agar sesuai dengan X.columns
new_patient = {
    'usia': 65,
    'jenis_kelamin': 1, # 1=Male
    'kolesterol': 280,
    'tekanan_darah': 160,
    'detak_jantung_maks': 140,
    'angina_saat_olahraga': 1 # 1=Yes
}

# Dapatkan nama kolom fitur dari data pelatihan (X)
feature_columns = X.columns.tolist()

print(f"Data Pasien Baru: {new_patient}")
prediction_result = predict_new_patient_heart_disease(model, new_patient, feature_columns)
print(f"\nHasil Prediksi: {prediction_result}")

# Contoh pasien lain
# Mengganti nama fitur ke Bahasa Indonesia agar sesuai dengan X.columns
new_patient_2 = {
    'usia': 40,
    'jenis_kelamin': 0, # 0=Female
    'kolesterol': 180,
    'tekanan_darah': 120,
    'detak_jantung_maks': 180,
    'angina_saat_olahraga': 0 # 0=No
}

print(f"\nData Pasien Baru 2: {new_patient_2}")
prediction_result_2 = predict_new_patient_heart_disease(model, new_patient_2, feature_columns)
print(f"Hasil Prediksi: {prediction_result_2}")


Data Pasien Baru: {'usia': 65, 'jenis_kelamin': 1, 'kolesterol': 280, 'tekanan_darah': 160, 'detak_jantung_maks': 140, 'angina_saat_olahraga': 1}

Hasil Prediksi: Pasien diprediksi memiliki **Penyakit Jantung** (Probabilitas: 99.71%)

Data Pasien Baru 2: {'usia': 40, 'jenis_kelamin': 0, 'kolesterol': 180, 'tekanan_darah': 120, 'detak_jantung_maks': 180, 'angina_saat_olahraga': 0}
Hasil Prediksi: Pasien diprediksi **Tidak Memiliki Penyakit Jantung** (Probabilitas: 99.96%)


## Form Input Interaktif untuk Pasien Baru

Anda dapat menggunakan formulir di bawah ini untuk memasukkan data pasien secara interaktif dan mendapatkan prediksi dari model. Coba ganti nilai-nilai fitur untuk melihat bagaimana prediksi model berubah.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.express as px

# Mendapatkan nama kolom fitur dari data pelatihan (X)
feature_columns_for_widgets = X.columns.tolist()

# Membuat widget input untuk setiap fitur
input_widgets = {}
for feature in feature_columns_for_widgets:
    if feature == 'jenis_kelamin':
        input_widgets[feature] = widgets.Dropdown(
            options=[('Pria', 1), ('Wanita', 0)], # 1=Pria, 0=Wanita
            description=feature.replace('_', ' ').title() + ':',
            value=1 # Default: Pria
        )
    elif feature == 'angina_saat_olahraga':
        input_widgets[feature] = widgets.Dropdown(
            options=[('Ya', 1), ('Tidak', 0)], # 1=Ya, 0=Tidak
            description=feature.replace('_', ' ').title() + ':',
            value=0 # Default: Tidak
        )
    else:
        # Menggunakan df_ml untuk mendapatkan median default yang sesuai dengan nama kolom baru
        default_value = df_ml[feature].median() if feature in df_ml.columns else 0
        input_widgets[feature] = widgets.IntText(
            description=feature.replace('_', ' ').title() + ':',
            value=int(default_value) # Default value as median
        )

# Tombol prediksi
predict_button = widgets.Button(description="Prediksi")

# Area output untuk menampilkan hasil prediksi dan visualisasi
output_prediction = widgets.Output()
output_plot = widgets.Output()

def on_predict_button_clicked(b):
    with output_prediction:
        clear_output()
        new_patient_input = {feature: widget.value for feature, widget in input_widgets.items()}

        print("Data Pasien Baru dari Input Form:")
        for k, v in new_patient_input.items():
            print(f"  {k.replace('_', ' ').title()}: {v}")

        # Convert new_patient_input to DataFrame for prediction
        new_patient_df = pd.DataFrame([new_patient_input], columns=feature_columns_for_widgets)

        # Perform prediction
        prediction = model.predict(new_patient_df)
        prediction_proba = model.predict_proba(new_patient_df)

        result_text = ""
        if prediction[0] == 1:
            result_text = f"Pasien diprediksi memiliki **Penyakit Jantung** (Probabilitas: {prediction_proba[0][1]*100:.2f}%)"
        else:
            result_text = f"Pasien diprediksi **Tidak Memiliki Penyakit Jantung** (Probabilitas: {prediction_proba[0][0]*100:.2f}%)"

        print(f"\nHasil Prediksi: {result_text}")

    # Calculate feature contributions for visualization
    # For logistic regression, contribution can be simplified as feature_value * coefficient
    # The coefficients are stored in model.coef_[0]
    feature_contributions = new_patient_df.iloc[0] * model.coef_[0]
    contribution_df = pd.DataFrame({
        'Fitur': feature_columns_for_widgets,
        'Kontribusi': feature_contributions.values
    }).sort_values(by='Kontribusi', ascending=True) # Sort to make plot more readable

    # Create an interactive bar chart for feature contributions
    fig_contributions = px.bar(
        contribution_df,
        x='Kontribusi',
        y='Fitur',
        orientation='h',
        title='<b>Kontribusi Fitur Terhadap Prediksi Pasien Ini</b>',
        labels={'Kontribusi': 'Dampak pada Log-Odds (Semakin Positif, Semakin Berisiko)', 'Fitur': 'Fitur Pasien'},
        color='Kontribusi',
        color_continuous_scale=px.colors.sequential.RdBu,
        template='plotly_white'
    )
    fig_contributions.update_layout(showlegend=False)

    with output_plot:
        clear_output()
        display(fig_contributions.show())

predict_button.on_click(on_predict_button_clicked)

# Tampilkan semua widget
print("Masukkan Data Pasien Baru:")
for feature in feature_columns_for_widgets:
    display(input_widgets[feature])
display(predict_button, output_prediction, output_plot)


Masukkan Data Pasien Baru:


IntText(value=53, description='Usia:')

Dropdown(description='Jenis Kelamin:', options=(('Pria', 1), ('Wanita', 0)), value=1)

IntText(value=227, description='Kolesterol:')

IntText(value=140, description='Tekanan Darah:')

IntText(value=160, description='Detak Jantung Maks:')

Dropdown(description='Angina Saat Olahraga:', index=1, options=(('Ya', 1), ('Tidak', 0)), value=0)

Button(description='Prediksi', style=ButtonStyle())

Output()

Output()